# Part B — Interannual Weather Sensitivity (Denmark)

1. **Section 1 (Optimisation)** — runs 9 weather-year optimisations, saves to `plots_part_b/`
2. **Section 2 (Plotting)** — loads from CSV, re-run freely without the solver

---
## Section 1 — Optimisation
*Run these cells once. Skip after CSVs exist.*

In [ ]:
import os
import pandas as pd
import numpy as np
import pypsa

df_elec = pd.read_csv('data/electricity_demand.csv', sep=';', index_col=0)
df_elec.index = pd.to_datetime(df_elec.index)

df_onshorewind = pd.read_csv('data/onshore_wind_1979-2017.csv', sep=';', index_col=0)
df_onshorewind.index = pd.to_datetime(df_onshorewind.index)

df_offshorewind = pd.read_csv('data/offshore_wind_1979-2017.csv', sep=';', index_col=0)
df_offshorewind.index = pd.to_datetime(df_offshorewind.index)

df_solar = pd.read_csv('data/pv_optimal.csv', sep=';', index_col=0)
df_solar.index = pd.to_datetime(df_solar.index)

country = 'DNK'

def annuity(n, r):
    return r / (1. - 1. / (1. + r) ** n) if r > 0 else 1 / n

In [ ]:
weather_years = [1979, 1985, 1991, 1995, 1999, 2005, 2010, 2013, 2015]
dict_results = {}

for year in weather_years:
    date_range = pd.date_range(f'{year}-01-01 00:00Z', f'{year}-12-31 23:00Z', freq='h')

    network = pypsa.Network()
    network.set_snapshots(date_range.values)
    network.add("Bus", "electricity bus")
    network.add("Load", "load", bus="electricity bus", p_set=df_elec[country].values)

    network.add("Carrier", "gas", co2_emissions=0.19)
    network.add("Carrier", "onshorewind")
    network.add("Carrier", "offshorewind")
    network.add("Carrier", "solar")

    CF_wind = df_onshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
    network.add("Generator", "Onshore wind", bus="electricity bus",
                p_nom_extendable=True, carrier="onshorewind",
                capital_cost=annuity(27, 0.07) * 1118775, marginal_cost=0,
                p_max_pu=CF_wind.values)

    CF_wind_off = df_offshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
    network.add("Generator", "Offshore wind", bus="electricity bus",
                p_nom_extendable=True, carrier="offshorewind",
                capital_cost=annuity(27, 0.07) * 2115944, marginal_cost=0,
                p_max_pu=CF_wind_off.values)

    CF_solar = df_solar[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
    network.add("Generator", "Solar", bus="electricity bus",
                p_nom_extendable=True, carrier="solar",
                capital_cost=annuity(25, 0.07) * 450000, marginal_cost=0,
                p_max_pu=CF_solar.values)

    network.add("Generator", "OCGT", bus="electricity bus",
                p_nom_extendable=True, carrier="gas",
                capital_cost=annuity(25, 0.07) * 453960, marginal_cost=30 / 0.41)

    network.add("Generator", "CCGT", bus="electricity bus",
                p_nom_extendable=True, carrier="gas",
                capital_cost=annuity(25, 0.07) * 880000, marginal_cost=30 / 0.56)

    network.optimize(solver_name='gurobi', solver_options={"OutputFlag": 0, "LogToConsole": 0})
    dict_results[year] = network.generators.p_nom_opt
    print(f"{year}: done")

df_results = pd.DataFrame(dict_results).T
print(df_results.div(1e3).round(2))

In [ ]:
os.makedirs('plots_part_b', exist_ok=True)
df_results.to_csv('plots_part_b/installed_capacities_weather_years.csv')
print("Saved to plots_part_b/installed_capacities_weather_years.csv")

---
## Section 2 — Plotting
*Start here after CSVs exist. No solver needed.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

df = pd.read_csv('plots_part_b/installed_capacities_weather_years.csv', index_col=0)

model_names = ['Onshore wind', 'Offshore wind', 'Solar', 'OCGT', 'CCGT']
labels      = ['Onshore wind', 'Offshore wind', 'Solar PV', 'Gas (OCGT)', 'Gas (CCGT)']
colors      = ['blue', 'dodgerblue', 'orange', 'crimson', 'darkviolet']

print(f"Loaded {len(df)} weather years")
df.div(1e3).round(2)

In [ ]:
x = np.arange(len(model_names))
X = np.tile(x, (len(df), 1)).ravel()
Y = df[model_names].to_numpy().ravel()
m, s = df[model_names].mean(), df[model_names].std()

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
ax.scatter(X, Y, s=35, alpha=0.7, label="Weather years")
ax.scatter(x, m.to_numpy(), marker="*", s=220, edgecolor="k", linewidth=0.8, label="Mean")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Installed capacity [MW]")
ax.grid(axis="y", alpha=0.3)
leg1 = ax.legend(loc="lower right", frameon=True)
ax.add_artist(leg1)
handles = [Line2D([], [], linestyle="none",
                  label=f"{tech}: {m[tech]:.0f}\u00b1{s[tech]:.0f} MW")
           for tech in model_names]
leg = ax.legend(handles=handles, loc="upper right", frameon=True, title="Mean \u00b1 std")
leg.get_title().set_fontweight("bold")
plt.title("Installed capacities across different weather years", fontweight="bold")
plt.tight_layout()
plt.savefig('plots_part_b/1b_installed_capacities_weather_years.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
df_gw = df[model_names].div(1e3)
x_pos = np.arange(len(df_gw))
width = 0.15

for i, (name, label, color) in enumerate(zip(model_names, labels, colors)):
    ax.bar(x_pos + i * width, df_gw[name], width, label=label, color=color, alpha=0.85)

ax.set_xticks(x_pos + width * 2)
ax.set_xticklabels(df.index.astype(int))
ax.set_xlabel('Weather year')
ax.set_ylabel('Installed capacity [GW]')
ax.set_title('Optimal capacity mix by weather year', fontweight='bold')
ax.legend(frameon=True, fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plots_part_b/1b_capacity_by_year.png', dpi=300, bbox_inches='tight')
plt.show()